# V11 Supplement: Frozen Tail-Gain and Stability Audit

## Purpose

V11 improved overall and high-stress prediction, but its validation top-1%
hotspot overlap was approximately 0.5997, just below the provisional 0.60
gate. It also increased maximum-stress prediction in a small number of cases.

This supplement therefore **does not run another symbolic search**. It freezes
the V10 formula, V11 mean calibration, V11 tail formula, scaling and RBF
centres, and evaluates only one transparent scalar:

`sigma(lambda) = sigma_v10 + delta_mu_case + lambda * scale_v10 * centered_tail_v11`

- `lambda = 0` is the V11 mean-calibrated V10 reference.
- `lambda = 1` is the current V11 formula.
- `lambda = 0.00...1.00` may be selected using validation cases only.
- `lambda = 1.05...1.20` is diagnostic only and cannot be selected.

Every element in every validation and internal-test case is evaluated. Each
case is checkpointed separately. The 50 final-test case files remain sealed.


## 1. Imports and package root


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "v11_gain_stability_audit.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NotebookCT3 package root.")


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from v11_gain_stability_audit import (
    V11GainAuditConfig,
    output_directory,
    preflight_v11_gain_audit,
    run_v11_gain_audit,
)

print("Package root:", PACKAGE_ROOT)
print("Python:", sys.executable)


Package root: /net/scratch/j96317yn/NotebookCT3
Python: /net/scratch/j96317yn/NotebookCT3/.venv/bin/python


## 2. Locked audit configuration


In [2]:
RUN_GAIN_AUDIT = True

CONFIG = V11GainAuditConfig(
    iteration=1,
    output_subdir="iteration_1",
    gain_min=0.00,
    gain_max=1.20,
    gain_step=0.05,
    max_selectable_gain=1.00,
    bootstrap_resamples=20_000,
    random_seed=42,
)

OUTPUT_DIR = output_directory(PACKAGE_ROOT, CONFIG)
display(pd.DataFrame([CONFIG.__dict__]).T.rename(columns={0: "value"}))
print("Gain grid:", CONFIG.gain_grid.tolist())
print("Output directory:", OUTPUT_DIR)


,value
iteration,1
output_subdir,iteration_1
gain_min,0.0
gain_max,1.2
gain_step,0.05
max_selectable_gain,1.0
bootstrap_resamples,20000
random_seed,42


Gain grid: [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2]
Output directory: /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1


## 3. Preflight and frozen provenance

This checks the existing 119/15/15/50 case split, confirms that V11 completed
without reading the final-test elements, loads the exact selected V11 formula,
and hashes all compact formula, scale, mean-model and RBF-centre artifacts.
The audit refuses to resume if any frozen input or configuration changes.


In [3]:
PREFLIGHT = preflight_v11_gain_audit(PACKAGE_ROOT, CONFIG)
display(PREFLIGHT["checks"])
display(PREFLIGHT["hash_audit"])
print("Validation cases:", len(PREFLIGHT["inputs"]["validation_ids"]))
print("Internal-test cases:", len(PREFLIGHT["inputs"]["internal_ids"]))
print("Sealed final-test cases:", len(PREFLIGHT["inputs"]["final_ids"]))
print("Audit signature:", PREFLIGHT["signature"]["audit_signature_sha256"])


,check,value,expected,pass
0,training_cases,119,119,True
1,validation_cases,15,15,True
2,internal_test_cases,15,15,True
3,sealed_final_cases,50,50,True
4,parent_v11_complete,complete,complete,True
5,parent_final_cases_read,0,0,True
6,selected_candidate_match,0,0,True
7,gain_grid_values,25,25,True
8,gain_one_selectable,True,True,True


,artifact,path,size_bytes,sha256
0,v11_completion,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,641,bb11dd0e5dd47d47052518986e6faf8883bb56a434aebc...
1,v11_composite,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,3887,fbf070549ceac74eb9f18510a0c46b6d0ed5f91a75e2fd...
2,v11_case_metrics,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,19814,cd750070d13215810723647e503489fa002edbdc3edab8...
3,v11_tail_formula,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,5005,a62e6005e4cbae190cbf07e22c55cdb56e0090ab4d19ee...
4,v11_tail_scaling,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,2831,eb04669f1a3d9426bbf60237965242ab6d53a97ad71f2e...
5,v11_rbf_centres,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,579,01116b0d3a988cc32a1fde8285c8d0da8e97d4ed8a0efd...
6,v11_mean_model,/net/scratch/j96317yn/NotebookCT3/outputs/11_t...,880,a1561a9a3222b432f74e0a268e409a89fa0a97e8a9391f...


Validation cases: 15
Internal-test cases: 15
Sealed final-test cases: 50
Audit signature: caf60fb4bf0034a9e0119d411d84bb840088aeb6fc26acc53bc7676d9f096676


## 4. Run or resume the audit

The validation scan is completed first and selects one gain without using the
internal-test set. Only after selection is frozen are `lambda=0`, `lambda=1`
and the selected gain evaluated on the internal-test cases. A completed case
is loaded from its small CSV checkpoint when this cell is re-run.


In [4]:
RESULT = None
if RUN_GAIN_AUDIT:
    RESULT = run_v11_gain_audit(
        PACKAGE_ROOT,
        CONFIG,
        preflight=PREFLIGHT,
    )
    display(pd.DataFrame([RESULT]))
else:
    print("Gain audit skipped because RUN_GAIN_AUDIT=False.")


[1/15] validation: full-case gain audit for case_18


[2/15] validation: full-case gain audit for case_29


[3/15] validation: full-case gain audit for case_45


[4/15] validation: full-case gain audit for case_51


[5/15] validation: full-case gain audit for case_54


...

[12/15] internal_test: full-case gain audit for case_159


[13/15] internal_test: full-case gain audit for case_196


[14/15] internal_test: full-case gain audit for case_197


[15/15] internal_test: full-case gain audit for case_199


,status,iteration,supplement_to,pySR_search_run,validation_cases,internal_test_cases,validation_gain_count,selected_gain,parent_v11_reproduction_checks,parent_v11_reproduction_pass,selection_pool_status,promotion_status,final_test_cases_read,audit_signature_sha256,elapsed_seconds,output_directory
0,complete,1,V11 tail-aware localised symbolic prototype,False,15,15,25,0.9,390,True,all_v11_promotion_gates,passes_all_v11_pilot_gates,0,caf60fb4bf0034a9e0119d411d84bb840088aeb6fc26ac...,69.26959,/net/scratch/j96317yn/NotebookCT3/outputs/12_v...


## 5. Saved evidence and interpretation


In [5]:
artifacts = {
    "completion": OUTPUT_DIR / "audit_complete.json",
    "selected_gain": OUTPUT_DIR / "selected_gain.json",
    "formula": OUTPUT_DIR / "selected_gain_formula.txt",
    "validation_scan": OUTPUT_DIR / "validation_gain_summary.csv",
    "split_metrics": OUTPUT_DIR / "selected_models_split_metrics.csv",
    "tail_adaptation": OUTPUT_DIR / "selected_models_tail_adaptation.csv",
    "paired_stability": OUTPUT_DIR / "paired_case_stability_summary.csv",
    "v11_reproduction": OUTPUT_DIR / "v11_reproduction_audit.csv",
    "worst_cases": OUTPUT_DIR / "worst_case_diagnostics.csv",
}
for name, path in artifacts.items():
    print(f"{name:18s} exists={path.exists()}  {path}")

if artifacts["selected_gain"].exists():
    print("\nSelected gain record")
    print(artifacts["selected_gain"].read_text(encoding="utf-8"))
if artifacts["validation_scan"].exists():
    scan = pd.read_csv(artifacts["validation_scan"])
    display(scan[
        [
            "gain",
            "selectable_gain",
            "macro_rmse",
            "mean_p95_relative_error",
            "mean_p99_underprediction_fraction",
            "mean_top1pct_hotspot_overlap",
            "max_prediction_abs_max_ratio",
            "all_v11_promotion_gates_pass",
            "engineering_selection_score",
            "selected_gain",
        ]
    ])
if artifacts["split_metrics"].exists():
    display(pd.read_csv(artifacts["split_metrics"]))
if artifacts["paired_stability"].exists():
    display(pd.read_csv(artifacts["paired_stability"]))
if artifacts["formula"].exists():
    print("\n" + artifacts["formula"].read_text(encoding="utf-8"))


completion         exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/audit_complete.json
selected_gain      exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/selected_gain.json
formula            exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/selected_gain_formula.txt
validation_scan    exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/validation_gain_summary.csv
split_metrics      exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/selected_models_split_metrics.csv
tail_adaptation    exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/selected_models_tail_adaptation.csv
paired_stability   exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/12_v11_gain_stability_audit/iteration_1/paired_case_stability_summary.csv
v11_r

,gain,selectable_gain,macro_rmse,mean_p95_relative_error,mean_p99_underprediction_fraction,mean_top1pct_hotspot_overlap,max_prediction_abs_max_ratio,all_v11_promotion_gates_pass,engineering_selection_score,selected_gain
0,0.00,True,1.793867,0.059518,0.139479,0.518198,1.010371,False,0.866667,False
1,0.05,True,1.782162,0.059126,0.138273,0.522161,1.017102,False,0.828571,False
2,0.10,True,1.771365,0.058695,0.137005,0.526407,1.023833,False,0.790476,False
3,0.15,True,1.761495,0.058177,0.135759,0.530220,1.030564,False,0.752381,False
4,0.20,True,1.752566,0.057595,0.134361,0.534682,1.037295,False,0.714286,False
5,0.25,True,1.744593,0.056793,0.132339,0.540859,1.044026,False,0.662857,False
6,0.30,True,1.737588,0.055648,0.130080,0.546437,1.050757,False,0.616190,False
7,0.35,True,1.731564,0.054369,0.127850,0.551282,1.057488,False,0.564762,False
8,0.40,True,1.726529,0.053371,0.125650,0.556827,1.064219,False,0.518095,False
9,0.45,True,1.722494,0.052293,0.122955,0.562637,1.070950,False,0.447619,False


,iteration,split,model,reporting_role,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,...,mean_top5_actual_rmse,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio
0,1,internal_test,v10_signed_staged_symbolic,parent_reference,15,6005400,1.334879,1.908915,0.571744,1.334879,...,3.724246,-2.177258,0.023138,0.007712,0.147758,0.141902,0.525917,0.509374,0.823726,1.444977
1,1,internal_test,v11_current_gain_1_00,current_v11,15,6005400,1.325252,1.860708,0.593101,1.325252,...,3.272094,-1.485711,0.052567,0.000000,0.111795,0.104480,0.581140,0.578788,0.881535,1.606682
2,1,internal_test,v11_mean_calibrated_v10,mean_calibration_only,15,6005400,1.342043,1.887214,0.581426,1.342043,...,4.003688,-2.620654,0.041996,0.041996,0.175932,0.171695,0.525917,0.509374,0.823726,1.429267
3,1,internal_test,v11_selected_gain_0.90,validation_selected,15,6005400,1.312537,1.844602,0.600115,1.312537,...,3.296655,-1.599205,0.040414,0.000485,0.124700,0.118171,0.588347,0.576757,0.882018,1.584976
4,1,validation,v10_signed_staged_symbolic,parent_reference,15,6005400,1.331691,1.919506,0.591093,1.331691,...,3.404932,-1.927182,0.076856,0.010752,0.126061,0.109526,0.508013,0.518198,0.826607,1.033887
5,1,validation,v11_current_gain_1_00,current_v11,15,6005400,1.239236,1.783323,0.647056,1.239236,...,2.976590,-1.283042,0.088691,0.000883,0.093968,0.076545,0.564279,0.599700,0.880037,1.144992
6,1,validation,v11_mean_calibrated_v10,mean_calibration_only,15,6005400,1.269426,1.833574,0.626885,1.269426,...,3.668645,-2.413445,0.059518,0.035247,0.143167,0.139479,0.508013,0.518198,0.826607,1.010371
7,1,validation,v11_selected_gain_0.90,validation_selected,15,6005400,1.228952,1.769763,0.652403,1.228952,...,2.990331,-1.396082,0.077506,0.001988,0.102731,0.088884,0.571915,0.600516,0.881652,1.131529


,split,comparison,selected_model,reference_model,metric,direction,n_paired_cases,mean_delta_selected_minus_reference,median_delta_selected_minus_reference,bootstrap_95ci_low,bootstrap_95ci_high,wilcoxon_statistic,wilcoxon_two_sided_p,descriptive_only
0,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,rmse,lower_is_better,15,-0.156747,-0.162706,-0.213051,-0.105487,2.0,0.000183,True
1,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,top5_actual_rmse,lower_is_better,15,-0.414602,-0.486099,-0.527437,-0.290361,2.0,0.000183,True
2,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,p95_relative_error,lower_is_better,15,0.000650,0.020994,-0.022791,0.021007,51.0,0.638672,True
3,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,p99_relative_error,lower_is_better,15,-0.023330,-0.028597,-0.032034,-0.013928,6.0,0.000854,True
4,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,p99_underprediction_fraction,lower_is_better,15,-0.020641,-0.027708,-0.027090,-0.013223,2.0,0.002366,True
5,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,top1pct_hotspot_overlap,higher_is_better,15,0.082318,0.084416,0.072811,0.090110,0.0,0.000653,True
6,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,top1_recall_in_predicted_top5,higher_is_better,15,0.055045,0.060689,0.045321,0.064236,0.0,0.000653,True
7,validation,selected_vs_v10,v11_selected_gain_0.90,v10_signed_staged_symbolic,prediction_abs_max_ratio,closer_to_one_is_better,15,0.079488,0.078134,0.073229,0.085983,0.0,0.000061,True
8,validation,selected_vs_current_v11,v11_selected_gain_0.90,v11_current_gain_1_00,rmse,lower_is_better,15,-0.013027,-0.010468,-0.017410,-0.009423,0.0,0.000061,True
9,validation,selected_vs_current_v11,v11_selected_gain_0.90,v11_current_gain_1_00,top5_actual_rmse,lower_is_better,15,0.013741,0.017705,-0.001633,0.027794,31.0,0.106995,True



CT3 frozen V11 gain-audited stress formula

Selected tail gain lambda = 0.90

sigma_v10 = (-0.0116301024532845*temperature_mean + 0.0470427355318997*temperature_p95 - 3.16087946558646*weight_loss_rate_mean + 0.103449215368151*z_max - 0.870195660324862*Abs(4.60760803334225*fluence_rate_p95 - 20.3572633494299) - 100.195330145459) + exp(-1.47091131932291*rho_mean - 1011.416708227*theta_sin_std - 1.1690770556861*weight_loss_rate_mean + 0.904578545888617*z_mean + 9.23869712445276) * ((1.10317833861391*(0.55673575*(0.0509611749551904*rho - 9.32241465638866)*(0.0509611749551904*rho - 8.66867535638866) - 31.9731949759449*(theta_cos - 0.922666019258146)**2)*(Abs(0.0509611749551904*rho - 8.59575422638866) - 2.4201684) + 0.285739561816426) + (0.0005298607274781943*z + 0.17657582192765694*(6.3559467282562*nearest_axial_boundary_fraction_proxy - 1.50570424883607)*(-7.28799975835273*nearest_radial_boundary_fraction_proxy + 0.0517386902634747*rho - 6.047025771972898) - 0.24332347927944745 + 0.793705

## Decision rule for V12

The selected gain remains a development result. Validation determines the
gain; internal test only checks whether its direction and worst-case behaviour
remain stable. The paired bootstrap intervals and Wilcoxon values are
descriptive because there are only 15 cases in each split.

The V12 design should be chosen only after examining:

1. whether any selectable gain passes all provisional V11 gates;
2. whether a gain below 1 reduces the maximum-stress overshoot without losing
   P99 and hotspot improvements;
3. whether gains above 1 improve hotspot overlap only by worsening maximum or
   P95 behaviour; and
4. whether the same trade-off is visible on the untouched internal-test cases.

No result in this notebook is final-test evidence.
